<a href="https://colab.research.google.com/github/427paul/AI_Agent/blob/main/%5BBDA%5D_10%EC%A3%BC%EC%B0%A8_%EC%8B%A4%EC%8A%B5_%EC%B5%9C%EC%A2%85_LCEL_BDA%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10주차 실습: 메모리 & 대화 관리
## 🏢 시나리오: 위니브마켓 고객 상담 챗봇 개발

```
여러분은 소규모 이커머스 스타트업 '위니브마켓'의 주니어 개발자입니다.
대표의 요청: "고객 상담 챗봇을 만들어주세요. 이름이랑 주문번호는 한 번만 입력하면
           대화 내내 기억해야 하고, 상담이 길어져도 느려지면 안 됩니다."
```

이번 주는 9주차에 배운 LCEL(`prompt | llm`)을 그대로 확장합니다.
새로운 프레임워크가 아니라 **메모리 래퍼 하나를 추가**하는 것입니다.

| 실습 블록 | 내용 | 셀 |
| --- | --- | --- |
| 사전 준비 | 환경 설정 | `0️⃣` |
| 원리 이해 | Stateless 문제 직접 확인 | `1️⃣~2️⃣` |
| Sprint 1 | RunnableWithMessageHistory 프로토타입 | `3️⃣~6️⃣` |
| Sprint 2 | 요약 체인으로 토큰 절약 | `7️⃣~9️⃣` |
| Sprint 3 | 위니봇 페르소나 + 다중 고객 처리 | `🔟~1️⃣3️⃣` |
| 비교 분석 | 셀프 체크 | `1️⃣4️⃣` |

> ⚠️ **시작 전 체크**: 아래 `0️⃣` 셀에서 API Key를 입력하세요.

---
## 0️⃣ 환경 설정

In [ ]:
# 패키지 설치 (Colab 최초 1회)
!pip install -q langchain langchain-google-genai
print('✅ 설치 완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 2.7 MB/s eta 0:00:00
✅ 설치 완료


In [ ]:
import os

# ──────────────────────────────────────────────
# ✏️  여기에 API Key 입력 (Google AI Studio 발급)
os.environ['GOOGLE_API_KEY'] = 'YOUR_GOOGLE_API'
# ──────────────────────────────────────────────

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.3,   # 고객 상담: 창의성보다 일관성이 중요
)
print('✅ LLM 준비 완료 — 위니브마켓 상담봇 개발 시작!')

✅ LLM 준비 완료 — 위니브마켓 상담봇 개발 시작!


---
## 🔬 원리 이해 — Stateless 문제를 직접 목격하기

### 1️⃣ 메모리 없이 상담하면?

In [ ]:
from langchain_core.messages import HumanMessage

print('[고객] 안녕하세요. 저는 김진환이고 배송 문의드려요.')
r1 = llm.invoke([HumanMessage(content='안녕하세요. 저는 김진환이고 배송 문의드려요.')])
print('[봇 ]', r1.content[:80])
print()

print('[고객] 제 이름이 뭐예요?')
r2 = llm.invoke([HumanMessage(content='제 이름이 뭐예요?')])
print('[봇 ]', r2.content[:80])
print()

print('⚠️  이름을 물어봤는데도 모른다고 합니다.')
print('   API는 매 호출을 완전히 새로운 요청으로 처리합니다 (Stateless).')

[고객] 안녕하세요. 저는 김진환이고 배송 문의드려요.
[봇 ] 안녕하세요, 김진환님. 배송 문의 주셔서 감사합니다.

정확한 배송 상황을 확인해 드리기 위해서는 몇 가지 정보가 필요합니다. 아래 정보를 알려

[고객] 제 이름이 뭐예요?
[봇 ] 저는 인공지능이기 때문에 사용자님의 개인 정보를 알 수 없습니다. 따라서 사용자님의 이름이 무엇인지 저에게는 정보가 없습니다.

혹시 저에게 이

⚠️  이름을 물어봤는데도 모른다고 합니다.
   API는 매 호출을 완전히 새로운 요청으로 처리합니다 (Stateless).


### 2️⃣ 해결책: 이전 대화를 함께 전달하기

In [ ]:
from langchain_core.messages import SystemMessage, AIMessage

messages = [
    SystemMessage(content='당신은 위니브마켓 고객 상담원입니다.'),
    HumanMessage(content='안녕하세요. 저는 김진환이고 배송 문의드려요.'),  # 이전 대화
    AIMessage(content='안녕하세요, 김진환 고객님! 어떤 문의사항이 있으신가요?'),  # 이전 대화
    HumanMessage(content='제 이름이 뭐예요?'),  # 현재 입력
]

response = llm.invoke(messages)
print('[봇]', response.content)
print()
print('💡 이전 대화를 포함시켰더니 이름을 기억합니다!')
print('   메모리가 하는 일 = 이 리스트를 매 호출마다 자동으로 쌓아서 전달하는 것')

[봇] 김진환 고객님이십니다.

💡 이전 대화를 포함시켰더니 이름을 기억합니다!
   메모리가 하는 일 = 이 리스트를 매 호출마다 자동으로 쌓아서 전달하는 것


---
## 🚀 Sprint 1: 프로토타입 — RunnableWithMessageHistory

> **미션**: 고객 이름과 주문번호를 기억하는 상담봇의 첫 버전을 만들어라.

```
설계 결정: 9주차 LCEL 체인(prompt | llm)에 메모리 래퍼만 추가
이유: 새로운 프레임워크 학습 없이 기존 패턴을 그대로 확장
```

### 3️⃣ MessagesPlaceholder — 이력이 들어갈 자리 예약

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 위니브마켓 고객 상담원입니다. 친절하게 응대하세요.'),
    MessagesPlaceholder(variable_name='history'),  # ← 대화 이력이 들어갈 자리
    ('human', '{input}'),
])

# 9주차와 동일한 LCEL 체인
chain = prompt | llm

print('✅ 프롬프트 + 체인 준비 완료')
print('   history 자리만 비어있는 상태 — 다음 셀에서 자동으로 채워집니다')

✅ 프롬프트 + 체인 준비 완료
   history 자리만 비어있는 상태 — 다음 셀에서 자동으로 채워집니다


### 4️⃣ RunnableWithMessageHistory로 메모리 연결

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 세션별 대화 이력을 저장할 딕셔너리
store = {}

def get_session_history(session_id: str):
    """session_id로 대화 이력을 가져옵니다. 처음이면 새로 만듭니다."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',        # 현재 입력이 들어갈 키
    history_messages_key='history',    # 이력이 들어갈 키 (Placeholder와 이름 일치)
)

print('✅ 메모리가 연결된 체인 준비 완료')

✅ 메모리가 연결된 체인 준비 완료


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### 5️⃣ 상담 진행 — session_id로 대화 이어가기

In [ ]:
config = {'configurable': {'session_id': 'CUST-001'}}

# 1턴: 고객 정보 접수
r1 = chain_with_history.invoke(
    {'input': '안녕하세요. 저는 김진환이고 주문번호는 WM-20241215-0042입니다. 배송 문의드려요.'},
    config=config,
)
print('[봇]', r1.content)
print()

# 2턴: 같은 session_id로 계속 — 자동으로 이전 대화가 이어짐
r2 = chain_with_history.invoke(
    {'input': '혹시 제 이름이랑 주문번호 다시 확인해주실 수 있어요?'},
    config=config,
)
print('[봇]', r2.content)

[봇] 안녕하세요, 김진환 고객님! 위니브마켓입니다. 😊

주문번호 WM-20241215-0042로 배송 문의 주셨네요.
고객님의 소중한 주문 건 배송 현황을 바로 확인해 드리겠습니다.

잠시만 기다려 주시면 확인 후 바로 안내해 드리겠습니다. 감사합니다!

[봇] 네, 김진환 고객님! 다시 한번 확인해 드리겠습니다.

고객님의 성함은 **김진환** 님이시고, 주문번호는 **WM-20241215-0042**가 맞으십니다.

정확하게 확인되었습니다. 혹시 다른 문의사항이 있으실까요?


### 6️⃣ 저장된 이력 직접 확인

In [ ]:
history = get_session_history('CUST-001')

print(f'저장된 메시지 개수: {len(history.messages)}개')
print()
for i, msg in enumerate(history.messages):
    role = '고객' if msg.type == 'human' else '봇'
    print(f'  [{i}] {role}: {msg.content[:40]}...')
print()
print('💡 이 리스트가 대화할수록 계속 쌓입니다.')
print('   상담이 길어지면 토큰이 늘어나는 문제 → Sprint 2에서 해결')

저장된 메시지 개수: 4개

  [0] 고객: 안녕하세요. 저는 김진환이고 주문번호는 WM-20241215-0042입니...
  [1] 봇: 안녕하세요, 김진환 고객님! 위니브마켓입니다. 😊

주문번호 WM-202...
  [2] 고객: 혹시 제 이름이랑 주문번호 다시 확인해주실 수 있어요?...
  [3] 봇: 네, 김진환 고객님! 다시 한번 확인해 드리겠습니다.

고객님의 성함은 ...

💡 이 리스트가 대화할수록 계속 쌓입니다.
   상담이 길어지면 토큰이 늘어나는 문제 → Sprint 2에서 해결


---
## ⚡ Sprint 2: 성능 개선 — 요약 체인으로 토큰 절약

> **미션**: 상담이 길어져도 응답 속도가 일정해야 한다.

```
설계 결정: '기존 요약 + 새 대화 → 새 요약' 을 만드는 별도 체인 추가
이유: RunnableWithMessageHistory는 이력 저장만 담당, 요약은 직접 구현해야 함
```

### 7️⃣ 요약 체인 만들기

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# 요약 전용 체인 — '기존 요약 + 새 대화' 를 받아 '새 요약'을 생성
summary_prompt = ChatPromptTemplate.from_messages([
    ('system', '아래 상담 요약과 새로운 대화를 한 문단으로 압축하세요. 고객 이름, 주문번호, 문의 핵심은 반드시 남기세요.'),
    ('human', '[기존 요약]\n{summary}\n\n[새 대화]\n고객: {user_input}\n상담원: {ai_output}'),
])

summary_chain = summary_prompt | llm | StrOutputParser()

def update_summary(summary: str, user_input: str, ai_output: str) -> str:
    """기존 요약 + 새 대화 → 새 요약"""
    return summary_chain.invoke({
        'summary': summary if summary else '(아직 대화 없음)',
        'user_input': user_input,
        'ai_output': ai_output,
    })

print('✅ 요약 체인 준비 완료')

✅ 요약 체인 준비 완료


### 8️⃣ 긴 상담 시뮬레이션 — 원문 누적 vs 요약본 비교

In [ ]:
import time

# 새 고객으로 별도 세션 시작 (Sprint 1의 CUST-001과 분리)
config_long = {'configurable': {'session_id': 'CUST-LONG'}}
current_summary = ''  # 누적 요약을 담을 변수

scenario = [
    '저는 김진환이고 주문번호는 WM-20241215-0042입니다.',
    '배송이 3일째 안 왔어요.',
    '주문할 때 주소는 서울시 마포구로 했어요.',
    '어제 배송기사님 연락을 못 받았어요.',
    '오늘 오후에 다시 연락이 올 수 있나요?',
]

print('5턴 상담 진행 중...')
for msg in scenario:
    # 원문은 RunnableWithMessageHistory가 그대로 누적
    response = chain_with_history.invoke({'input': msg}, config=config_long)

    # 요약본은 별도로 직접 갱신
    current_summary = update_summary(current_summary, msg, response.content)

    time.sleep(1)  # Free Tier 분당 제한 대응

print()
original_history = get_session_history('CUST-LONG')
original_text_length = sum(len(m.content) for m in original_history.messages)

print('📊 메모리 크기 비교')
print(f'원문 누적 (RunnableWithMessageHistory) : {original_text_length:,} 글자')
print(f'요약본만 유지                          : {len(current_summary):,} 글자')
print()
print('[현재 요약 내용]')
print(current_summary)

5턴 상담 진행 중...

📊 메모리 크기 비교
원문 누적 (RunnableWithMessageHistory) : 1,127 글자
요약본만 유지                          : 300 글자

[현재 요약 내용]
김진환 고객은 주문번호 WM-20241215-0042의 서울시 마포구 주소로의 배송이 3일째 지연되고 있으며, 어제 배송 기사님으로부터 연락을 받지 못했다고 문의했다. 이에 오늘 오후에 다시 연락이 올 수 있는지 문의했고, 상담원은 배송 기사님이 다음 영업일에 재배송을 시도하거나 고객에게 다시 연락을 조율하는 경우가 많으므로 오늘 연락 가능성이 충분하다고 안내하며, 해당 주문의 배송 현황을 다시 한번 자세히 조회하여 오늘 재배송이 예정되어 있는지 또는 배송 기사님이 언제쯤 다시 연락을 주실 예정인지 확인 후 안내해 주겠다고 밝혔다.


### 9️⃣ 요약본만으로 핵심 정보가 유지되는지 확인

In [ ]:
# 요약본만 시스템 컨텍스트로 넣어서 질문 — 원문 history 없이도 답할 수 있는지 확인
check_prompt = f'[이전 상담 요약]\n{current_summary}\n\n[현재 질문]\n제 이름이랑 주문번호가 뭐였죠?'

check_response = llm.invoke(check_prompt)
print('[질문] 제 이름이랑 주문번호가 뭐였죠? (요약본만 사용)')
print('[답변]', check_response.content)
print()
print('💡 원문 5턴 전체를 보내지 않고도, 압축된 요약본만으로 핵심 정보가 유지됩니다.')

[질문] 제 이름이랑 주문번호가 뭐였죠? (요약본만 사용)
[답변] 네, 고객님의 성함은 **김진환**님이시고요, 주문번호는 **WM-20241215-0042**입니다.

💡 원문 5턴 전체를 보내지 않고도, 압축된 요약본만으로 핵심 정보가 유지됩니다.


---
## 🏭 Sprint 3: 운영 배포 — 위니봇 페르소나 + 다중 고객 처리

> **미션**: 100명이 동시에 상담해도 대화가 섞이면 안 된다.
>          '위니봇'이라는 명확한 페르소나도 필요하다.

> ⚠️ **주의**: 시스템 프롬프트 안에 `{ }` (중괄호)를 쓰면 LangChain이 변수로
> 잘못 인식할 수 있습니다. 일반 문장에서는 중괄호 사용을 피하세요.

### 🔟 위니봇 페르소나 + 새 체인 구성

In [ ]:
# 페르소나 프롬프트 — 중괄호 없이 일반 문장으로 작성
WINIVBOT_PROMPT = '''당신은 위니브마켓의 AI 고객 상담원 위니봇입니다.

핵심 규칙:
1. 고객이 이름을 알려주면 이후 대화에서 반드시 고객님이라는 호칭과 함께 이름을 불러주세요.
2. 주문번호를 알려주면 기억하고 관련 문의 시 함께 언급하세요.
3. 해결 불가한 문제는 담당자 연결을 제안하세요.
4. 답변은 2~3문장으로 간결하게, 항상 친절하게 응대하세요.'''

winivbot_prompt = ChatPromptTemplate.from_messages([
    ('system', WINIVBOT_PROMPT),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}'),
])

winivbot_chain = winivbot_prompt | llm

winivbot_with_history = RunnableWithMessageHistory(
    winivbot_chain,
    get_session_history,   # Sprint 1에서 만든 함수 재사용
    input_messages_key='input',
    history_messages_key='history',
)

print('✅ 위니봇 체인 구성 완료')

✅ 위니봇 체인 구성 완료


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### 1️⃣1️⃣ 상담 함수 + 다중 고객 동시 처리

In [ ]:
def consult(customer_id: str, message: str) -> str:
    """위니봇과 상담하는 함수. customer_id로 고객을 구분합니다."""
    response = winivbot_with_history.invoke(
        {'input': message},
        config={'configurable': {'session_id': customer_id}},
    )
    print(f'고객({customer_id}): {message}')
    print(f'위니봇       : {response.content}')
    print()
    return response.content

print('--- 고객 A 상담 시작 ---')
consult('WINIV-A', '안녕하세요. 김진환이고 주문번호 WM-20241215-0042예요.')
consult('WINIV-A', '배송이 3일째 안 왔어요.')

print('--- 고객 B 상담 시작 (A와 동시) ---')
consult('WINIV-B', '안녕하세요. 이민수입니다. 주문번호 WM-20241214-0031 환불하고 싶어요.')
consult('WINIV-B', '환불은 며칠 걸려요?')

print('--- 고객 A 상담 재개 ---')
consult('WINIV-A', '오늘 중으로 배송 가능한가요?')

--- 고객 A 상담 시작 ---
고객(WINIV-A): 안녕하세요. 김진환이고 주문번호 WM-20241215-0042예요.
위니봇       : 안녕하세요, 김진환 고객님! 위니봇입니다. 주문번호 WM-20241215-0042로 접수되셨습니다. 무엇을 도와드릴까요?

고객(WINIV-A): 배송이 3일째 안 왔어요.
위니봇       : 김진환 고객님, 주문번호 WM-20241215-0042의 배송이 3일째 지연되고 있어 불편하시겠어요. 현재 배송 상황을 확인해 드리겠습니다. 잠시만 기다려 주세요.

--- 고객 B 상담 시작 (A와 동시) ---
고객(WINIV-B): 안녕하세요. 이민수입니다. 주문번호 WM-20241214-0031 환불하고 싶어요.
위니봇       : 이민수님, 주문번호 WM-20241214-0031 환불 요청 확인했습니다. 환불 절차를 위해 상품 수령 여부와 환불 사유를 알려주시면 빠르게 도와드리겠습니다.

고객(WINIV-B): 환불은 며칠 걸려요?
위니봇       : 이민수님, 주문번호 WM-20241214-0031의 환불은 반품 상품이 저희에게 도착하여 확인된 후 영업일 기준 3~5일 이내에 처리됩니다. 카드사나 은행에 따라 실제 환불 금액이 반영되기까지는 며칠 더 소요될 수 있습니다.

--- 고객 A 상담 재개 ---
고객(WINIV-A): 오늘 중으로 배송 가능한가요?
위니봇       : 김진환 고객님, 주문번호 WM-20241215-0042의 오늘 중 배송 여부는 제가 직접 확인해 드리기 어렵습니다. 정확한 배송 일정 확인을 위해 담당자 연결을 도와드릴까요?



'김진환 고객님, 주문번호 WM-20241215-0042의 오늘 중 배송 여부는 제가 직접 확인해 드리기 어렵습니다. 정확한 배송 일정 확인을 위해 담당자 연결을 도와드릴까요?'

1️⃣3️⃣ 전체 세션 현황 조회 (운영 모니터링)

In [ ]:
print('=== 위니브마켓 상담 세션 현황 ===')
print(f'활성 세션 수: {len(store)}개')
print()

for session_id, history in store.items():
    msgs = history.messages
    turns = len(msgs) // 2
    print(f'📁 {session_id}')
    print(f'   대화 턴수: {turns}턴')
    print(f'   최근 메시지: {msgs[-1].content[:40] if msgs else "없음"}...')
    print()

print('💡 실제 서비스에서는 store를 DB로 교체해')
print('   앱 재시작 후에도 대화를 이어갈 수 있습니다.')

=== 위니브마켓 상담 세션 현황 ===
활성 세션 수: 4개

📁 CUST-001
   대화 턴수: 2턴
   최근 메시지: 네, 김진환 고객님! 다시 한번 확인해 드리겠습니다.

고객님의 성함은 ...

📁 CUST-LONG
   대화 턴수: 5턴
   최근 메시지: 김진환 고객님, 오늘 오후에 다시 연락이 올 수 있는지 궁금하시군요. 네...

📁 WINIV-A
   대화 턴수: 3턴
   최근 메시지: 김진환 고객님, 주문번호 WM-20241215-0042의 오늘 중 배송 ...

📁 WINIV-B
   대화 턴수: 2턴
   최근 메시지: 이민수님, 주문번호 WM-20241214-0031의 환불은 반품 상품이 ...

💡 실제 서비스에서는 store를 DB로 교체해
   앱 재시작 후에도 대화를 이어갈 수 있습니다.


---
## 📊 오늘 배운 내용 정리

### 셀프 체크

In [ ]:
checklist = [
    ('LLM이 Stateless인 이유를 설명할 수 있다',
     '매 API 호출은 독립적. 이전 대화를 직접 전달해야 기억함'),
    ('MessagesPlaceholder의 역할을 설명할 수 있다',
     '프롬프트 안에 대화 이력이 삽입될 위치를 예약하는 자리표시자'),
    ('RunnableWithMessageHistory가 하는 일을 설명할 수 있다',
     '9주차 LCEL 체인을 감싸서 이력 저장/불러오기를 자동화'),
    ('요약 체인이 왜 별도로 필요한지 설명할 수 있다',
     'RunnableWithMessageHistory는 저장만 담당, 요약은 직접 구현해야 함'),
    ('session_id로 사용자를 분리하는 이유를 설명할 수 있다',
     '같은 체인을 써도 고객별로 독립된 대화 이력을 유지하기 위해'),
]

print('📋 오늘 학습 핵심 개념 셀프 체크')
for i, (q, a) in enumerate(checklist, 1):
    print(f'\n{i}. {q}')
    print(f'   → {a}')

print()
print('='*50)
print('다음 주 (11주차): Agent & Tool')
print('메모리가 있는 챗봇에 검색·계산 도구를 붙이면?')
print('→ LLM이 스스로 판단해서 도구를 쓰는 Agent를 배웁니다!')

📋 오늘 학습 핵심 개념 셀프 체크

1. LLM이 Stateless인 이유를 설명할 수 있다
   → 매 API 호출은 독립적. 이전 대화를 직접 전달해야 기억함

2. MessagesPlaceholder의 역할을 설명할 수 있다
   → 프롬프트 안에 대화 이력이 삽입될 위치를 예약하는 자리표시자

3. RunnableWithMessageHistory가 하는 일을 설명할 수 있다
   → 9주차 LCEL 체인을 감싸서 이력 저장/불러오기를 자동화

4. 요약 체인이 왜 별도로 필요한지 설명할 수 있다
   → RunnableWithMessageHistory는 저장만 담당, 요약은 직접 구현해야 함

5. session_id로 사용자를 분리하는 이유를 설명할 수 있다
   → 같은 체인을 써도 고객별로 독립된 대화 이력을 유지하기 위해

다음 주 (11주차): Agent & Tool
메모리가 있는 챗봇에 검색·계산 도구를 붙이면?
→ LLM이 스스로 판단해서 도구를 쓰는 Agent를 배웁니다!
